# 02 — Clean & Standardize

Load the raw chart into DuckDB and clean it **in SQL** (workspace norm):
strip `$`/commas, cast to numeric, derive `decade` and the `inflation_multiple`
(adjusted / nominal). Result: the `films_adjusted` table.

In [ ]:
import sys, os
from pathlib import Path
PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

import pandas as pd
from io import StringIO
from src.ingest import load_config
from src.clean_quality import get_connection, load_to_duckdb, run_sql, register_source, save_interim

cfg = load_config('config.yaml')
con = get_connection(cfg)

# Reload the raw HTML parsed in 01 into a raw DuckDB table.
html = (Path(cfg['paths']['data_raw']) / 'bom_top_lifetime_adjusted_2022.html').read_text(encoding='utf-8')
raw = pd.read_html(StringIO(html))[0]
load_to_duckdb(raw, 'bom_raw', con)
print('bom_raw:', con.execute('SELECT COUNT(*) FROM bom_raw').fetchone()[0], 'rows')

## Load CPI-U and set the “today's dollars” base year
Load the CPI-U annual averages ingested in 01 (FRED CPIAUCNS) into DuckDB.
The **base year** for constant dollars is the latest COMPLETE annual CPI (the
final year is partial and excluded). Every nominal gross below is converted
with `nominal × (cpi_base / cpi_release_year)` — one consistent CPI method
across all charts, replacing Box Office Mojo's ticket-price/2022 figures.

In [ ]:
# Per-year CPI table (integer years) for the per-film join.
cpi = pd.read_parquet(Path(cfg['paths']['data_raw']) / 'cpi_annual.parquet')
load_to_duckdb(cpi, 'cpi', con)

# Base index = TRAILING 12 MONTHS of CPI-U (most current honest 'today\u2019s
# dollars'; the latest calendar year is incomplete due to CPI reporting lag).
_m = pd.read_csv(Path(cfg['paths']['data_raw']) / 'cpi_cpiaucns_monthly.csv')
_m.columns = ['date','cpi']; _m['date'] = pd.to_datetime(_m['date'])
_last12 = _m.dropna(subset=['cpi']).sort_values('date').tail(12)
CPI_BASE = float(_last12['cpi'].mean())
CPI_BASE_LABEL = f"12 months to {_last12['date'].max():%b %Y}"
print(f"CPI base = trailing 12 months (mean CPI={CPI_BASE:.1f}); today\u2019s-dollars label: '{CPI_BASE_LABEL}'")

## Clean in DuckDB
Money strings (`$1,895,421,694`) become `BIGINT`; `decade` uses integer
division; `inflation_multiple` = adjusted / nominal (how many times the film's
original take the adjusted figure represents).

In [ ]:
films = run_sql(f'''
    SELECT
        CAST(b."Rank" AS INTEGER)                                   AS rank_adjusted,
        b."Title"                                                   AS title,
        CAST(REGEXP_REPLACE(b."Adj. Lifetime Gross", '[$,]', '', 'g') AS BIGINT) AS adjusted_gross_bom,
        CAST(REGEXP_REPLACE(b."Lifetime Gross",       '[$,]', '', 'g') AS BIGINT) AS nominal_gross,
        CAST(b."Est. Num Tickets" AS BIGINT)                        AS est_tickets,
        CAST(b."Year" AS INTEGER)                                   AS release_year,
        (CAST(b."Year" AS INTEGER) // 10) * 10                      AS decade,
        -- CPI 'today's dollars' (the consistent basis used by the charts):
        ROUND(CAST(REGEXP_REPLACE(b."Lifetime Gross", '[$,]', '', 'g') AS DOUBLE)
              * {CPI_BASE} / c.cpi)::BIGINT                         AS adjusted_gross,
        ROUND({CPI_BASE} / c.cpi, 2)                              AS inflation_multiple
    FROM bom_raw b
    LEFT JOIN cpi c ON c.year = CAST(b."Year" AS INTEGER)
    ORDER BY adjusted_gross DESC
''', con)
load_to_duckdb(films, 'films_adjusted', con)
print(films.shape, '| base:', CPI_BASE_LABEL)
films[['title','release_year','nominal_gross','adjusted_gross','adjusted_gross_bom']].head(10)

## Quick sanity checks

In [ ]:
# Classics should dominate a CPI-adjusted board; the smallest inflation
# multiples belong to the most recent films (little inflation since release).
print('Top adjusted (today\u2019s $):', films.iloc[0]['title'], films.iloc[0]['release_year'])
print('No nulls in key cols:',
      films[['adjusted_gross','nominal_gross','est_tickets','release_year']].notna().all().all())
films[['title','release_year','inflation_multiple']].sort_values('inflation_multiple').head(5)

## Clean the WORLDWIDE chart (domestic vs international split)
Same money-string cleaning; a bare `-` (no gross in a market) becomes NULL.
Produces `films_worldwide` with domestic/foreign/worldwide gross + the split.

In [ ]:
ww_html = (Path(cfg['paths']['data_raw']) / 'bom_ww_top_lifetime.html').read_text(encoding='utf-8')
ww_raw = pd.read_html(StringIO(ww_html))[0]
load_to_duckdb(ww_raw, 'bom_ww_raw', con)
ww = run_sql('''
    WITH cleaned AS (
        SELECT CAST("Rank" AS INTEGER) AS rank_worldwide, "Title" AS title,
               CAST("Year" AS INTEGER) AS release_year,
               TRY_CAST(NULLIF(REGEXP_REPLACE("Worldwide Lifetime Gross",'[$,]','','g'),'-') AS BIGINT) AS worldwide_gross,
               TRY_CAST(NULLIF(REGEXP_REPLACE("Domestic Lifetime Gross",'[$,]','','g'),'-') AS BIGINT) AS domestic_gross,
               TRY_CAST(NULLIF(REGEXP_REPLACE("Foreign Lifetime Gross",'[$,]','','g'),'-') AS BIGINT) AS foreign_gross
        FROM bom_ww_raw)
    SELECT rank_worldwide, title, worldwide_gross, domestic_gross, foreign_gross, release_year,
        ROUND(100.0*domestic_gross/NULLIF(worldwide_gross,0),1) AS domestic_pct,
        ROUND(100.0*foreign_gross /NULLIF(worldwide_gross,0),1) AS foreign_pct
    FROM cleaned ORDER BY worldwide_gross DESC''', con)
load_to_duckdb(ww, 'films_worldwide', con)
print('most international (lowest domestic %):')
print(ww.nsmallest(5,'domestic_pct')[['title','release_year','worldwide_gross','domestic_pct']].to_string(index=False))
ww.head()

## Clean the FOREIGN-LANGUAGE chart (foreign films at the U.S. box office)
Box Office Mojo's Foreign Language genre chart: non-English-language films
ranked by **domestic (U.S. & Canada) lifetime gross** (NOMINAL $). Strip the
`$`/commas and cast to BIGINT. Produces `films_foreign_us` — the universe for
the “foreign films that broke into the U.S.” chart. Unlike the earlier
worldwide-subset approach, this captures a film's U.S. take regardless of its
worldwide total, so U.S.-favored foreign hits (Parasite, Crouching Tiger,
Life Is Beautiful, the Bollywood titles) are all present.

In [ ]:
fl_html = (Path(cfg['paths']['data_raw']) / 'bom_foreign_language.html').read_text(encoding='utf-8')
fl_raw = pd.read_html(StringIO(fl_html))[0]
load_to_duckdb(fl_raw, 'bom_foreign_raw', con)

# Origin country (primary), ingested from TMDB in 01. Map ISO-3166 -> name.
fc = pd.read_parquet(Path(cfg['paths']['data_raw']) / 'tmdb_foreign_countries.parquet')
ISO_TO_NAME = {'HK':'Hong Kong','IT':'Italy','JP':'Japan','CN':'China','KR':'South Korea',
    'MX':'Mexico','FR':'France','IN':'India','AR':'Argentina','ES':'Spain','DE':'Germany',
    'TW':'Taiwan','TH':'Thailand','US':'United States','GB':'United Kingdom','SE':'Sweden',
    'BR':'Brazil','RU':'Russia','DK':'Denmark','NO':'Norway','CA':'Canada','AU':'Australia'}
fc['origin_name'] = fc['origin_country'].map(ISO_TO_NAME).fillna(fc['origin_country'])
load_to_duckdb(fc[['title','release_year','origin_country','origin_name']], 'foreign_countries', con)

foreign_us = run_sql(f'''
    WITH base AS (
      SELECT
        CAST(b."Rank" AS INTEGER)                                          AS rank_foreign,
        b."Title"                                                          AS title,
        CAST(REGEXP_REPLACE(b."Lifetime Gross", '[$,]', '', 'g') AS BIGINT) AS domestic_gross_nominal,
        TRY_CAST(b."Release Date"[-4:] AS INTEGER)                         AS release_year,
        b."Distributor"                                                    AS distributor
      FROM bom_foreign_raw b)
    SELECT base.rank_foreign, base.title, base.domestic_gross_nominal,
        ROUND(base.domestic_gross_nominal * {CPI_BASE} / c.cpi)::BIGINT     AS domestic_gross,
        base.release_year, base.distributor, fc.origin_country, fc.origin_name
    FROM base
    LEFT JOIN cpi c ON c.year = base.release_year
    LEFT JOIN foreign_countries fc
      ON fc.title = base.title AND fc.release_year = base.release_year
    ORDER BY domestic_gross DESC
''', con)
load_to_duckdb(foreign_us, 'films_foreign_us', con)
print(foreign_us.shape, '| CPI-adjusted to', CPI_BASE_LABEL)
foreign_us[['rank_foreign','title','domestic_gross_nominal','domestic_gross','release_year','origin_name']].head(15)

## Clean the DOMESTIC all-time top 1000 + origin (films created outside the U.S.)
Load the 5 domestic pages, parse gross, and join TMDB origin country. The key
definition: **`is_foreign` = the film's PRIMARY origin country is not the U.S.**

Why primary origin and not the `is_us` flag: `is_us` is True whenever the U.S.
appears *anywhere* in a film's origin/production countries, so it marks nearly
every big co-production (Harry Potter, James Bond, LOTR) as U.S. and would hide
them. TMDB's first-listed origin country is the discriminating field — it puts
the culturally British franchises under GB, which is the honest answer to
“created outside the U.S.” Co-productions are a documented judgment call
(recorded in SOURCES.md), not a hidden one. Produces `films_domestic_all`.

In [ ]:
dom_frames = [pd.read_html(StringIO((Path(cfg['paths']['data_raw']) / f'bom_domestic_all_{o:04d}.html').read_text(encoding='utf-8')))[0]
              for o in cfg['sources']['bom_domestic_all']['offsets']]
dom_all = pd.concat(dom_frames, ignore_index=True)
load_to_duckdb(dom_all, 'bom_domestic_raw', con)

dc = pd.read_parquet(Path(cfg['paths']['data_raw']) / 'tmdb_domestic_countries.parquet')
dc['origin_name'] = dc['origin_country'].map(ISO_TO_NAME).fillna(dc['origin_country'])
load_to_duckdb(dc[['title','release_year','origin_country','origin_name']], 'domestic_countries', con)

# Production-company proxy for the 'Hollywood industry' cut (ingested in 01).
cc = pd.read_parquet(Path(cfg['paths']['data_raw']) / 'tmdb_domestic_companies.parquet')
load_to_duckdb(cc[['title','release_year','companies_str','company_countries','has_us_studio']], 'domestic_companies', con)

films_domestic_all = run_sql(f'''
    SELECT
        CAST(b."Rank" AS INTEGER)                                          AS rank_domestic,
        b."Title"                                                          AS title,
        CAST(REGEXP_REPLACE(b."Lifetime Gross", '[$,]', '', 'g') AS BIGINT) AS us_gross_nominal,
        ROUND(CAST(REGEXP_REPLACE(b."Lifetime Gross", '[$,]', '', 'g') AS DOUBLE)
              * {CPI_BASE} / c.cpi)::BIGINT                                 AS us_gross,
        CAST(b."Year" AS INTEGER)                                          AS release_year,
        dc.origin_country, dc.origin_name,
        (dc.origin_country IS NOT NULL AND dc.origin_country <> 'US')       AS is_foreign,
        co.companies_str, co.company_countries, co.has_us_studio,
        (co.has_us_studio IS NOT NULL AND co.has_us_studio = FALSE)         AS is_non_hollywood
    FROM bom_domestic_raw b
    LEFT JOIN cpi c ON c.year = CAST(b."Year" AS INTEGER)
    LEFT JOIN domestic_countries dc
      ON dc.title = b."Title" AND dc.release_year = CAST(b."Year" AS INTEGER)
    LEFT JOIN domestic_companies co
      ON co.title = b."Title" AND co.release_year = CAST(b."Year" AS INTEGER)
    ORDER BY us_gross DESC
''', con)
load_to_duckdb(films_domestic_all, 'films_domestic_all', con)
n_for = int(films_domestic_all['is_foreign'].sum())
n_nh = int(films_domestic_all['is_non_hollywood'].sum())
print(f'{len(films_domestic_all)} domestic films; {n_for} non-US primary origin; {n_nh} non-Hollywood; CPI-adjusted to {CPI_BASE_LABEL}')
films_domestic_all[films_domestic_all['is_foreign']][['rank_domestic','title','release_year','us_gross','origin_name']].head(15)

## Load the genre data (ingested from TMDB in 01)

The TMDB genre lookup was done in `01-ingest` (that's where external sources are
pulled — see the API-key note there). Here we just **load** the ingested genre
records from `data/raw/tmdb_genres.parquet` into DuckDB — no API calls in the
cleaning stage. Builds `films_genre` (one row per film) and `film_genres_long`
(one row per film x genre, which drives the genre market-split analysis).

In [ ]:
genre_raw = pd.read_parquet(Path(cfg['paths']['data_raw']) / 'tmdb_genres.parquet')
country_raw = pd.read_parquet(Path(cfg['paths']['data_raw']) / 'tmdb_countries.parquet')

films_genre = (genre_raw.drop(columns=['genres']).rename(columns={'year':'release_year'})
               .merge(country_raw, on='tmdb_id', how='left')
               .drop_duplicates(['title','release_year']))  # one row per film
load_to_duckdb(films_genre, 'films_genre', con)

long_rows = [{'title': r['title'], 'release_year': r['year'], 'genre': g}
             for r in genre_raw.to_dict('records') for g in r['genres']]
film_genres_long = pd.DataFrame(long_rows).drop_duplicates()
load_to_duckdb(film_genres_long, 'film_genres_long', con)
print(f'{films_genre["matched"].sum()}/{len(films_genre)} matched; '
      f'{int(films_genre["is_us"].sum())} US-made; {len(film_genres_long)} film-genre rows')
films_genre['primary_genre'].value_counts().head(8)

## Save interim + register provenance

In [ ]:
save_interim(films, cfg, 'films_adjusted.parquet')
save_interim(ww, cfg, 'films_worldwide.parquet')
save_interim(films_genre, cfg, 'films_genre.parquet')
save_interim(film_genres_long, cfg, 'film_genres_long.parquet')
save_interim(foreign_us, cfg, 'films_foreign_us.parquet')
save_interim(films_domestic_all, cfg, 'films_domestic_all.parquet')
save_interim(cpi, cfg, 'cpi_annual.parquet')

_cpi_note = f'All grosses CPI-adjusted to constant {CPI_BASE_LABEL} dollars (today\u2019s dollars) via CPI-U (FRED CPIAUCNS); nominal kept alongside.'
register_source(con, 'films_adjusted',
    name='Box Office Mojo - Top Lifetime Grosses (domestic) + CPI adjustment',
    url=cfg['sources']['bom_adjusted']['url'], license='Data (c) IMDb/Box Office Mojo; CPI via FRED/BLS.',
    notes='Domestic (US/Canada) lifetime gross. adjusted_gross = nominal in constant today-dollars via CPI-U; adjusted_gross_bom kept for reference (BOM ticket-price, 2022).',
    methodology=_cpi_note + ' Lifetime totals include re-releases.',
    series_breaks='CPI method differs from BOM ticket-price method; figures will not match BOM adjusted chart by design.')
register_source(con, 'films_worldwide',
    name='Box Office Mojo - Top Lifetime Grosses (Worldwide)',
    url=cfg['sources']['bom_worldwide']['url'], license='Data (c) IMDb/Box Office Mojo.',
    notes='Worldwide/domestic/foreign lifetime gross (NOMINAL $) + split. Domestic = US & Canada. (Shares are ratios, so not CPI-adjusted.)',
    methodology='Studio-reported theatrical receipts; lifetime totals include re-releases.',
    series_breaks='Nominal dollars; worldwide totals favor recent wide-release films.')
register_source(con, 'films_genre',
    name='TMDB - film genres', url='https://www.themoviedb.org/',
    license='TMDB API; non-commercial attribution.',
    notes='Genre(s) per film matched by title+year. Multi-genre; used as a ratio in the market split.',
    methodology='TMDB /search/movie by title+year, first result.', series_breaks='')
register_source(con, 'films_foreign_us',
    name='Box Office Mojo - Top Lifetime Grosses (Foreign Language genre) + CPI adjustment',
    url=cfg['sources']['bom_foreign']['url'], license='Data (c) IMDb/Box Office Mojo; CPI via FRED/BLS.',
    notes='Non-English-language films by DOMESTIC (US/Canada) lifetime gross. domestic_gross = CPI-adjusted today-dollars; domestic_gross_nominal kept. LANGUAGE-based: includes some US-produced non-English films; misses English-language non-US films.',
    methodology=_cpi_note + ' Genre keyword = primary language not English.',
    series_breaks='Language grouping, not country of origin.')
register_source(con, 'films_domestic_all',
    name='Box Office Mojo - Top Lifetime Grosses (Domestic top 1000) + TMDB origin + CPI',
    url=cfg['sources']['bom_domestic_all']['url'], license='Data (c) IMDb/Box Office Mojo; origin via TMDB; CPI via FRED/BLS.',
    notes='Domestic top-1000 by lifetime gross (us_gross = CPI today-dollars; us_gross_nominal kept), enriched with TMDB primary origin. is_foreign = primary origin != US; is_non_hollywood = no US studio among producers.',
    methodology=_cpi_note + ' is_foreign follows TMDB origin_country (co-productions like Harry Potter/Bond read non-US). is_non_hollywood = no US-registered production company (~13 films). Documented proxies; no ground-truth industry-of-origin field exists.',
    series_breaks='2026 titles still in theaters have incomplete totals (snapshot as of Sep 2026).')
register_source(con, 'cpi',
    name=cfg['sources']['cpi']['source_name'], url=cfg['sources']['cpi']['csv_url'],
    license='Public domain (US BLS via FRED).',
    notes=f'CPI-U annual averages (from monthly), 1913-present. Base year for constant dollars = {CPI_BASE_LABEL} (latest complete year).',
    methodology='Annual average of monthly CPIAUCNS; final partial year flagged and not used as base.',
    series_breaks='')
print(con.execute('SELECT duckdb_table, source_name FROM _sources ORDER BY duckdb_table').df().to_string(index=False))

---
**Next:** `03-prepare.ipynb` packages the three exports + codebooks.

## Cleanup
Close the DuckDB connection so the write lock is released.

In [ ]:
con.close()
print('connection closed')